In [1]:
import pandas as pd
from pandas import DataFrame
import numpy as np

In [2]:
import requests
import random
import xlrd
import csv
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
# formatting for decimal places
pd.set_option("display.float_format", "{:.2f}".format)

In [4]:
# setting seed for model reproducibility
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)

In [5]:
# setting the destination for the data folder
path = os.path.join(os.getcwd(), "../data")
norm_path = os.path.normpath(path)

In [6]:

from io import BytesIO


In [7]:

def download_xlsx_to_csv(month_list, base_url, output_dir="csv_output"):
    """
    Downloads .xlsx files from URLs and converts them to .csv files.

    Parameters:
    ----------
    month_list : list
        Example: ["2015_05", "2015_06", "2015_07"]

    base_url : str
        Example: "https://www.dmr.nd.gov/oilgas/mpr/"

    output_dir : str
        Folder where CSV files will be saved.
    """

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    for month in month_list:
        try:
            # Construct file URL
            file_url = f"{base_url}{month}.xlsx"

            print(f"Downloading: {file_url}")

            # Download file
            response = requests.get(file_url, timeout=30)
            response.raise_for_status()

            # Read Excel file into pandas dataframe
            excel_data = BytesIO(response.content)

            # Read first sheet
            df = pd.read_excel(excel_data, engine="openpyxl")

            # Output CSV path
            csv_filename = os.path.join(output_dir, f"{month}.csv")

            # Save as CSV
            df.to_csv(csv_filename, index=False)

            print(f"Saved CSV: {csv_filename}")

        except Exception as e:
            print(f"Failed for {month}: {e}")



In [8]:
train_list = ["2015_05","2015_06","2015_07","2015_08","2015_09","2015_10","2015_11","2015_12",
    "2016_01","2016_02","2016_03","2016_04","2016_05","2016_06","2016_07","2016_08","2016_09","2016_10","2016_11","2016_12",
    "2017_01","2017_02","2017_03","2017_04","2017_05","2017_06","2017_07","2017_08","2017_09","2017_10","2017_11","2017_12",
    "2018_01","2018_02","2018_03","2018_04","2018_05","2018_06","2018_07","2018_08","2018_09","2018_10","2018_11","2018_12"]

In [9]:
website = "https://www.dmr.nd.gov/oilgas/mpr/"

In [ ]:

download_xlsx_to_csv(train_list, website, output_dir='train_csv_output')

In [10]:
import glob

In [11]:

def combine_csv_files(input_dir="csv_output",
                      output_file="combined_data.csv"):
    """
    Combines all CSV files in a folder into a single CSV.

    Parameters:
    ----------
    input_dir : str
        Folder containing monthly CSV files.

    output_file : str
        Name/path of the final combined CSV file.
    """

    # Get all CSV files
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))

    if not csv_files:
        print("No CSV files found.")
        return

    dataframes = []

    for file in csv_files:
        try:
            print(f"Reading: {file}")

            # Read CSV
            df = pd.read_csv(file)

            # Optional: Add source month column
            month_name = os.path.basename(file).replace(".csv", "")
            df["source_month"] = month_name

            dataframes.append(df)

        except Exception as e:
            print(f"Failed to read {file}: {e}")

    # Combine all dataframes
    combined_df = pd.concat(dataframes, ignore_index=True)

    # Save combined CSV
    combined_df.to_csv(output_file, index=False)

    print(f"\nCombined CSV saved as: {output_file}")
    print(f"Total rows: {len(combined_df)}")


In [ ]:

# Example usage
combine_csv_files(
    input_dir="csv_output",
    output_file="training_combined.csv"
)

In [12]:
test_list = ["2019_01","2019_02","2019_03","2019_04","2019_05","2019_06","2019_07","2019_08","2019_09","2019_10","2019_11","2019_12"]

In [ ]:
download_xlsx_to_csv(test_list, website, output_dir="test_csv_output")

In [27]:
combine_csv_files(
    input_dir="test_csv_output",
    output_file="testing_combined.csv"
)

No CSV files found.


In [22]:
# ARPS Decline Curve Analysis

def pre_process(df, column):
    #df.drop("Unnamed: 0", axis=1, inplace=True)
    df.info()
    print(df.columns)
    # descriptive statistics
    df.describe().T
    df.head(15)
    df.nunique()    
    df.dtypes
    df.shape
    # filtering
    df.dropna(inplace=True)
    # drop rows where oil rate is 0
    df = df[(df[column].notnull()) & (df[column] > 0)]
    return df


In [14]:
# visualization/plotting libraries
import matplotlib as mpl
import matplotlib.style
import seaborn as sns  
import matplotlib.pyplot as plt
# setting to default parameters
plt.rcParams.update(plt.rcParamsDefault)

In [15]:
# matplotlib settings
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use('seaborn-v0_8-white')
mpl.rcParams["figure.figsize"] = (12, 8)
mpl.rcParams["axes.grid"] = False

In [16]:


def plot_production_rate(df):
    '''Plot decline curve using production rates'''
    sns.lineplot(x = df['ReportDate'], y = df['oil_rate'], markers=True, dashes=False, 
                 label="Oil Production",color='blue', linewidth=1.5)
    plt.title('Decline Curve', fontweight='bold', fontsize = 20)
    plt.xlabel('Time', fontweight='bold', fontsize = 15)
    plt.ylabel('Oil Production Rate (bbl/d)', fontweight='bold', fontsize = 15)
    plt.show()



In [17]:

def decline_curve(curve_type, q_i):
    if curve_type == "exponential":

        def exponential_decline(T, d):
            return q_i * np.exp(-d * T)
        return exponential_decline

    elif curve_type == "hyperbolic":

        def hyperbolic_decline(T, d_i, b):
            return q_i / np.power((1 + b * d_i * T), 1.0 / b)
        return hyperbolic_decline

    elif curve_type == "harmonic":

        def parabolic_decline(T, d_i):
            return q_i / (1 + d_i * T)
        return parabolic_decline

    else:
        raise "Unknown Decline Curve!"


In [18]:


def L2_norm(Q, Q_obs):
    return np.sum(np.power(np.subtract(Q, Q_obs), 2))

In [ ]:
# reading train and test data
train_prod = pd.read_csv('training_combined.csv')
test_prod = pd.read_csv("testing_combined.csv")

In [ ]:

# Basic Processing and data exploration
train_prod = pre_process(train_prod,column="Oil")
test_prod = pre_process(test_prod, column="Oil")